`requirements/gpu.txt`

In [2]:
import pandas as pd, gc, rustworkx as rx
from qiskit_aer.noise import NoiseModel
from qiskit_aer import AerSimulator
from qiskit_experiments.library import QuantumVolume
from collections import defaultdict
from calibration_helpers import load_payload

In [3]:
def noisy_simulator_from_payload(payload):
    '''Creates a noisy AerSimulator from the noise model and coupling map in the payload.'''
    
    noise_model = NoiseModel.from_dict(payload["noise_model"])
    coupling_map = payload.get("coupling_map")
    return AerSimulator(
        noise_model=noise_model,
        coupling_map=coupling_map,
        basis_gates=noise_model.basis_gates,
        method="automatic",
        device="GPU",
    )


def run_qv_for_subset(backend, ideal_backend, subset, shots=2000, trials=100, seed=42):
    '''Runs the Quantum Volume experiment for a given subset of qubits and returns the results.'''

    exp = QuantumVolume(
        physical_qubits=list(subset),
        trials=trials,
        seed=seed,
        simulation_backend=ideal_backend,
    )

    exp.set_run_options(shots=shots)

    exp.set_transpile_options(
        coupling_map=backend.coupling_map,
        layout_method="trivial",
        routing_method="basic",
        optimization_level=0,
        seed_transpiler=seed,
        num_processes=0,
    )

    return exp.run(backend).block_for_results()

In [ ]:
def connected_n_tuples_from_coupling(coupling_map, n: int):
    """
    Returns list of n-tuples (sorted tuples) that are connected (induced subgraph is connected).
    """

    nodes = sorted({int(a) for a, _ in coupling_map} | {int(b) for _, b in coupling_map})
    if n == 1:
        return [(q,) for q in nodes]
    if len(nodes) < n:
        return []

    # build mapping: qubit label -> rustworkx node index
    g = rx.PyGraph(multigraph=False)
    label_to_idx = {}
    for q in nodes:
        label_to_idx[q] = g.add_node(q)  # store the label as node payload

    # add edges
    for a, b in coupling_map:
        a = int(a); b = int(b)
        g.add_edge(label_to_idx[a], label_to_idx[b], None)

    # enumerate connected k-node subgraphs (returns lists of node indices)
    subgraphs = rx.connected_subgraphs(g, n)

    # convert node indices back to qubit labels (payloads), sort each tuple
    out = []
    for idxs in subgraphs:
        labels = sorted(g[idx] for idx in idxs)
        out.append(tuple(labels))

    # ensure uniqueness + deterministic ordering
    return sorted(set(out))

In [5]:
def run_qv_over_connected_n_tuples(calibration_path, n: int, shots=2000, trials=100, seed=42):
    '''Runs Quantum Volume experiments over all connected n-tuples of qubits and returns a summary of results.'''

    payload = load_payload(calibration_path)
    coupling_map = payload.get("coupling_map")

    tuples_ = connected_n_tuples_from_coupling(coupling_map, n)

    backend = noisy_simulator_from_payload(payload)
    backend.set_options(batched_shots_gpu=True)
    ideal_backend = AerSimulator(method="statevector", device="GPU")

    summaries = []
    for i, subset in enumerate(tuples_, 1):
        print(f"[{i}/{len(tuples_)}] running QV on subset {subset}")

        exp_data = run_qv_for_subset(
            backend=backend,
            ideal_backend=ideal_backend,
            subset=subset,
            shots=shots,
            trials=trials,
            seed=seed,
        )

        df = exp_data.analysis_results(dataframe=True)
        hop_row = df[df["name"] == "mean_HOP"].iloc[0]

        mean_hop = hop_row["value"].nominal_value
        hop_err  = hop_row["value"].std_dev

        summaries.append((subset, mean_hop, hop_err))

        del exp_data, df
        gc.collect()

    return summaries

In [ ]:
# from qiskit_experiments.framework import BatchExperiment
# from qiskit_experiments.library import QuantumVolume

# def build_qv_exp(backend, ideal_backend, subset, shots, trials, seed):
#     exp = QuantumVolume(
#         physical_qubits=list(subset),
#         trials=trials,
#         seed=seed,
#         simulation_backend=ideal_backend,
#     )
#     exp.set_run_options(shots=shots)
#     exp.set_transpile_options(
#         coupling_map=backend.coupling_map,
#         layout_method="trivial",
#         routing_method="basic",
#         optimization_level=0,
#         seed_transpiler=seed,
#     )
#     return exp

# def run_qv_batched(calibration_path, n, shots=2000, trials=100, seed=42):
#     payload = load_payload(calibration_path)
#     coupling_map = payload.get("coupling_map")
#     tuples_ = connected_n_tuples_from_coupling(coupling_map, n)

#     backend = noisy_simulator_from_payload(payload)
#     backend.set_options(batched_shots_gpu=True)

#     ideal_backend = AerSimulator(method="statevector")  # see note below

#     exps = [
#         build_qv_exp(backend, ideal_backend, subset, shots, trials, seed)
#         for subset in tuples_
#     ]

#     batched = BatchExperiment(exps, backend=backend, flatten_results=True)
#     expdata = batched.run().block_for_results()
#     return tuples_, expdata

Pick one 3Q subset and one 4Q subset and print the transpiled circuits. Look for: swap, cx count, depth. You'll likely see much higher relative routing overhead for 3Q.

In [ ]:
# circ = exp.circuits()[0]
# tcirc = transpile(circ, backend)
# print(tcirc.count_ops())

Leaving the `layout_method`, `routing_method`, and `optimization_level`, the transpiler may be able to route through qubits outside the subset, which defeats the purpose of comparing subsets as self-contained candidate patches. Maybe should pass the induced coupling map on that subset, not the full backend coupling map. 

```python
exp.set_transpile_options(
        coupling_map=sub_cmap,
        layout_method="sabre",
        routing_method="sabre",
        optimization_level=1,
        seed_transpiler=seed,
    )
```

In [6]:
cal_path = "calibrations/ibm_marrakesh/20260129_101824.json"

qubits = 6

expdata = run_qv_over_connected_n_tuples(
    cal_path,
    n=qubits,
    shots=100,
    trials=100,
    seed=42,
)

df = pd.DataFrame(expdata, columns=["subset", "mean_HOP", "hop_error"])
df = df.sort_values("mean_HOP", ascending=False).reset_index(drop=True)

out_csv = f"results/ibm_marrakesh_qv_{qubits}_tuples.csv"
df.to_csv(out_csv, index=False)

/tmp/ipykernel_6177/2068522216.py:4: DeprecationWarning: from_dict has been deprecated as of qiskit-aer 0.15.0 and will be removed no earlier than 3 months from that release date.
  noise_model = NoiseModel.from_dict(payload["noise_model"])


[1/926] running QV on subset (0, 1, 2, 3, 4, 5)
[2/926] running QV on subset (0, 1, 2, 3, 4, 16)
[3/926] running QV on subset (0, 1, 2, 3, 16, 23)
[4/926] running QV on subset (1, 2, 3, 4, 5, 6)
[5/926] running QV on subset (1, 2, 3, 4, 5, 16)
[6/926] running QV on subset (1, 2, 3, 4, 16, 23)
[7/926] running QV on subset (1, 2, 3, 16, 22, 23)
[8/926] running QV on subset (1, 2, 3, 16, 23, 24)
[9/926] running QV on subset (2, 3, 4, 5, 6, 7)
[10/926] running QV on subset (2, 3, 4, 5, 6, 16)
[11/926] running QV on subset (2, 3, 4, 5, 16, 23)
[12/926] running QV on subset (2, 3, 4, 16, 22, 23)
[13/926] running QV on subset (2, 3, 4, 16, 23, 24)
[14/926] running QV on subset (2, 3, 16, 21, 22, 23)
[15/926] running QV on subset (2, 3, 16, 22, 23, 24)
[16/926] running QV on subset (2, 3, 16, 23, 24, 25)
[17/926] running QV on subset (3, 4, 5, 6, 7, 8)
[18/926] running QV on subset (3, 4, 5, 6, 7, 16)
[19/926] running QV on subset (3, 4, 5, 6, 7, 17)
[20/926] running QV on subset (3, 4, 5, 6, 

7Q: 1591 $\approx$ 16h

8Q: 2800 $\approx$ 32h

9Q: 5054 $\approx$ 64h

---

10Q: 9286 $\approx$ 128h

11Q: 17343 $\approx$ 256h

---

12Q: 32617 $\approx$ 512h

13Q: 61477 $\approx$ 1024h 

Another important aspect of a benchmark for quantum computers is that the noise profile on the device changes over time which means that benchmarks on a specific part of the hardware may not be consistent over time. Another future research area is to quantify the correlation between QV and error metrics such as aggregate error, cross-talk, and average qubit fidelity.